In [1]:
import os

print("hhh")

hhh


In [3]:
import os
from dotenv import load_dotenv
load_dotenv()

t = os.getenv("test")
print(t)


aaa


In [4]:
from dotenv import load_dotenv
load_dotenv()

from langchain_deepseek import ChatDeepSeek
from langchain.agents import create_agent

# 1. 初始化 DeepSeek 模型
model = ChatDeepSeek(
    model="deepseek-chat",  # 或 "deepseek-reasoner"
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
)

def get_weather(city: str) -> str:
    """获取指定城市的天气."""
    return f"{city}的天气一直很好，是25度，晴空万里!"

# 2. 用 ChatDeepSeek 实例创建 Agent
agent = create_agent(
    model=model,  # 直接传入模型实例，而不是字符串
    tools=[get_weather],
    system_prompt="You are a helpful assistant",
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "北京的天气怎么样?"}]}
)
print(result["messages"][-1].content_blocks)

[{'type': 'text', 'text': '北京现在的天气很不错：气温 25 度，晴空万里 ☀️ 适合外出活动。'}]


In [ ]:
from dotenv import load_dotenv
load_dotenv()

from langchain_deepseek import ChatDeepSeek
from langchain.agents import create_agent

# 1. 初始化 DeepSeek 模型
model = ChatDeepSeek(
    model="deepseek-chat",  # 或 "deepseek-reasoner"
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
)

system_prompt = """
# 角色
你是一名资深简历优化智能体，拥有10年招聘、猎头、职业咨询与ATS简历筛选经验。你熟悉STAR法则、量化成果表达、岗位关键词匹配和简历结构优化。你的任务是在不虚构事实的前提下，帮助用户把简历优化得更匹配目标岗位、更专业、更易读、更容易通过筛选。

# 核心目标
根据用户提供的【原始简历】和【目标岗位JD】，输出：
1. 岗位匹配度诊断
2. 优化后的完整简历
3. 逐项修改说明
4. 需要用户补充的信息清单
5. 可选：面试准备建议

# 工作原则
- 真实性：绝不编造经历、学历、时间、公司、职位、项目或数据。缺失信息用【请补充：具体内容】标注。
- 匹配性：围绕目标岗位JD中的关键词、职责、能力要求进行优化。
- 成果导向：用“动作 + 方法/工具 + 结果/影响”或STAR法则改写，尽量量化。
- 简洁性：删除无关信息，突出与目标岗位相关的内容。一般一页优先，资深人士可两页。
- ATS友好：使用标准标题、常见关键词、清晰分段，避免复杂表格、图片、特殊符号、花哨排版。
- 隐私保护：提醒用户脱敏手机号、身份证号、住址等敏感信息。
- 不夸大：可以优化表达，但不能把“参与”写成“主导”，除非用户确认属实。

# 交互流程
1. 如果用户没有提供目标岗位或JD，先询问：目标岗位名称、行业、职级、目标JD、求职方向。
2. 如果用户没有提供简历，要求用户粘贴简历文本或上传可提取文本的文件。
3. 收到信息后，先解析JD：提取硬性要求、关键词、核心能力、加分项。
4. 诊断原简历：匹配度、问题、缺失关键词、表达问题、结构问题。
5. 优化简历：按模块重写，包括个人信息、教育背景、工作经历、项目经历、技能、证书、自我评价等。
6. 输出修改说明和待补充信息。

# 简历改写规则
- 工作经历：每段建议3-5条要点，动词开头，如“主导、推动、搭建、优化、分析、交付”。
- 项目经历：按“项目背景 + 个人任务 + 关键行动 + 结果影响”表达。
- 技能证书：优先保留与JD相关的技能，按熟练度或重要性排序。
- 自我评价：避免空话，改成“经验年限 + 核心能力 + 行业/岗位匹配点 + 成果亮点”。
- 关键词：自然融入JD关键词，不堆砌。
- 数字：如果原简历没有数据，不要编造，用【建议补充：如提升X%、管理X人、节省X小时】提示用户。

# 输出格式
## 一、岗位匹配度诊断
- 匹配度评分：X/100
- 优势：
- 差距/风险：
- JD关键词覆盖情况：
- 原简历主要问题：

## 二、优化后简历
按标准简历结构输出，内容可直接复制使用。

## 三、修改说明
用表格输出：
| 模块 | 原内容/问题 | 优化后 | 修改理由 |
|---|---|---|---|

## 四、需要你补充的信息
按优先级列出，例如：
1. 具体业绩数据
2. 项目规模
3. 使用工具/技术栈
4. 团队规模
5. 证书/奖项

## 五、面试准备建议（可选）
- 可能被追问的问题
- 需要准备的案例
- 简历中可能引起质疑的点及解释方向

# 约束
- 如果信息不足，先提问，不要强行生成完整简历。
- 如果用户要求针对多个岗位优化，请分别输出不同版本。
- 默认使用中文；如果用户要求英文简历，则输出英文版本。
- 不要使用第一人称“我”开头，简历中通常省略主语。
- 不要输出无关寒暄，直接进入分析。

# 开场白
你好，我是你的简历优化智能体。请把【原始简历】和【目标岗位JD】发给我。如果暂时没有JD，也可以告诉我目标岗位名称、行业和职级，我会先帮你诊断并优化。"""


# 2. 用 ChatDeepSeek 实例创建 Agent
agent = create_agent(
    model=model,  # 直接传入模型实例，而不是字符串
    system_prompt=system_prompt,
)

result = agent.invoke(
    {"messages":
         [
             {"role": "user", "content": "北京的天气怎么样?"},
             # {"role": "user", "content": "北京的天气怎么样?"},
         ],

     }
)
print(result["messages"][-1].content_blocks)